# 02 — Exploratory Data Analysis

**Owner:** Seneviratne's lane — distributions, outliers, correlations, class-conditional comparisons, `grip_lost` analysis.

Viva questions to be ready for: Are outliers errors or signal? Which features separate the classes?

Remember: every important choice gets a decision-log cell below it (what we decided, evidence, alternative considered, why rejected).

In [ ]:
import sys
from pathlib import Path

# Jupyter's kernel CWD is notebooks/, so add the repo root to sys.path
# before importing anything from src/.
sys.path.append(str(Path.cwd().parent))

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.config import RANDOM_STATE, FIGURES_DIR
from src.pipeline import load_data

pd.set_option("display.max_columns", None)
sns.set_theme(style="whitegrid")

In [ ]:
df = load_data()
df.shape

## Sensor families

TODO: group columns by family for reuse below (Current_*, Temperature_*, Speed_J*, Tool_current).

In [ ]:
current_cols = [c for c in df.columns if c.startswith("Current_")]
temperature_cols = [c for c in df.columns if c.startswith("Temperature_")]
speed_cols = [c for c in df.columns if c.startswith("Speed_")]
tool_cols = ["Tool_current"]

current_cols, temperature_cols, speed_cols, tool_cols

## Distributions by sensor family and class

TODO: histogram/KDE per feature, split by `Robot_ProtectiveStop` (0 vs 1), grouped by family. Save the key ones to `FIGURES_DIR / "fig02_distributions_<family>.png"`.

In [ ]:
# TODO: e.g. for col in current_cols: sns.kdeplot(data=df, x=col, hue="Robot_ProtectiveStop")

## Boxplots + outlier counts

TODO: boxplot per feature; count IQR-based outliers per feature (below Q1 - 1.5*IQR or above Q3 + 1.5*IQR). Decide per-feature whether outliers look like sensor errors or genuine extreme readings (part of the viva answer).

In [ ]:
def iqr_outlier_count(series):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    return ((series < lower) | (series > upper)).sum()

numeric_cols = current_cols + temperature_cols + speed_cols + tool_cols
pd.Series({c: iqr_outlier_count(df[c]) for c in numeric_cols}).sort_values(ascending=False)

In [ ]:
# TODO: boxplots, e.g. df[current_cols].plot(kind="box")

## Correlation heatmap

TODO: correlation matrix across all sensor features (+ target). Save to `FIGURES_DIR / "fig02_correlation.png"`.

In [ ]:
# TODO: corr = df[numeric_cols + ["Robot_ProtectiveStop"]].corr(); sns.heatmap(corr, ...)

## grip_lost cross-tab

TODO: `pd.crosstab(df["grip_lost"], df["Robot_ProtectiveStop"])`. Plan expects grip_lost overlaps a stop on only 3 rows out of 243 grip-loss rows — confirm it reads as a weak, largely separate signal rather than a strong predictor.

In [ ]:
pd.crosstab(df["grip_lost"], df["Robot_ProtectiveStop"])

## Class-conditional comparisons

TODO: for each sensor family, compare summary stats (median/mean) between stop rows and normal rows — which features separate the classes most clearly? This feeds directly into notebook 03's leakage discussion, so cross-check findings with that notebook (e.g. speed features separating the classes is the leakage signal, not necessarily a usable feature).

In [ ]:
df.groupby("Robot_ProtectiveStop")[numeric_cols].median().T

## Decision log

### Decision: <what we decided>
- **Evidence:** <figure/number>
- **Alternative considered:** <...>
- **Why rejected:** <...>